In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg,udf
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", "Engineering", 75000, 25),
    (2, "Bob", "Marketing", 60000, 30),
    (3, "Charlie", "Engineering", 80000, 35),
    (4, "David", "Sales", 65000, 28),
    (5, "Eve", "Marketing", 70000, 32)
]
columns = ["Id", "Name", "Department", "Salary", "Age"]
df = spark.createDataFrame(data, columns);

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:59:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [65]:
df_select=df.select("Name","Department","Salary")

In [66]:
df_select.show()

+-------+-----------+------+
|   Name| Department|Salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|    Bob|  Marketing| 60000|
|Charlie|Engineering| 80000|
|  David|      Sales| 65000|
|    Eve|  Marketing| 70000|
+-------+-----------+------+



In [67]:
df_filter=df.filter(col("Salary")>65000)

In [68]:
df_filter.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  3|Charlie|Engineering| 80000| 35|
|  5|    Eve|  Marketing| 70000| 32|
+---+-------+-----------+------+---+



In [69]:
df_with_col=df.withColumn("Bonus",col("Salary")*0.10)

In [70]:
df_with_col.show()

+---+-------+-----------+------+---+------+
| Id|   Name| Department|Salary|Age| Bonus|
+---+-------+-----------+------+---+------+
|  1|  Alice|Engineering| 75000| 25|7500.0|
|  2|    Bob|  Marketing| 60000| 30|6000.0|
|  3|Charlie|Engineering| 80000| 35|8000.0|
|  4|  David|      Sales| 65000| 28|6500.0|
|  5|    Eve|  Marketing| 70000| 32|7000.0|
+---+-------+-----------+------+---+------+



In [71]:
df.write.mode("overwrite").csv("output/employeeTable",header=True)

In [72]:
df.toPandas().to_csv(
    "output/employeeTableNew.csv",
    index=False
)

In [73]:
df=spark.read \
    .option("header",True) \
    .csv("output/employeeTable")

In [74]:
df.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  3|Charlie|Engineering| 80000| 35|
|  4|  David|      Sales| 65000| 28|
|  5|    Eve|  Marketing| 70000| 32|
|  1|  Alice|Engineering| 75000| 25|
|  2|    Bob|  Marketing| 60000| 30|
+---+-------+-----------+------+---+



In [75]:
df.createOrReplaceTempView("employees")

In [76]:
result=spark.sql("""SELECT * FROM employees WHERE Salary>70000""").show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  3|Charlie|Engineering| 80000| 35|
|  1|  Alice|Engineering| 75000| 25|
+---+-------+-----------+------+---+



In [77]:
df=spark.range(0,1000000).withColumn("value",col("id")%1000)

In [78]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [79]:
print("before Partition",df.rdd.getNumPartitions())

before Partition 2


In [80]:
df_repartion=df.repartition(10)

In [81]:
print(df_repartion.rdd.getNumPartitions())

[Stage 63:>                                                         (0 + 2) / 2]

10


In [82]:
df_colesced=df_repartion.coalesce(2)

In [83]:
print(df_colesced.rdd.getNumPartitions())

[Stage 64:>                                                         (0 + 2) / 2]

2


[Stage 64:=============================>                            (1 + 1) / 2]

In [ ]:
df_repartion=df.repartition(20)

In [ ]:
print("repartition:",df_repartion.rdd.getNumPartitions())

In [ ]:
df_repartion.write.mode("overwrite").csv("output/employeeRepPartioTable",header=True)

In [ ]:
df=spark.range(0,10000000).withColumn("value",col("id")%1000)

In [ ]:
optimised_df=df.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [ ]:
optimised_df.explain()

In [ ]:
import time
start_time=time.time()
count_uncached=optimised_df.count()
end_time=time.time()
print(f"1. optimised Execution |Cout:{count_uncached}| time:{round(end_time-start_time,4)}second")

In [ ]:
optimised_df.cache()

In [ ]:
start_time=time.time()
count_cached=optimised_df.count()
end_time=time.time()
print(f"1. first cached Execution |Cout:{count_cached}| time:{round(end_time-start_time,4)}second")

In [ ]:
start_time=time.time()
count_cached=optimised_df.count()
end_time=time.time()
print(f"1. second cached Execution |Cout:{count_cached}| time:{round(end_time-start_time,4)}second")

In [8]:
data=[("Alice",25),("charlie",16),("Nitesh",18)]
colums=["name","age"]

In [11]:
df1=spark.createDataFrame(data,colums)

In [12]:
df1.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|charlie| 16|
| Nitesh| 18|
+-------+---+



In [13]:
def catorize(age):
    if age>=18:
        return "can Be drive"
    return "can not be drive"

In [14]:
dl_category=udf(catorize,StringType())

NameError: name 'StringType' is not defined